# 🏗️ Notebook 1: ChatGPT — Requirements & Architecture


## 🛠️ Setup

```bash
cd 06-system-designs/chatgpt
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

All code in this lab is **self-contained Python** — stdlib only, no GPUs, no model weights, no
Docker. We simulate the scheduler, the KV cache, the streaming protocol and the cost model in
memory, because the hard parts of this design are queueing, batching, streaming and money —
none of which need a container.


## 💡 What we're designing

An **LLM-backed chat product**: a user types a message, a large language model reads the whole
conversation so far, and streams back a reply token by token.

From the outside it looks like a chat app. It is not. The difference is one number: serving a
Discord message costs a fraction of a microsecond of CPU; serving one ChatGPT turn costs
**several GPU-seconds**. Every design decision in this lab falls out of that.

Three properties make it hard:

1. **The unit of work is huge and variable.** One turn can be 200 tokens or 20,000, and you
   don't know which until it's over.
2. **The response is a stream, not a value.** The user starts reading before the server has
   finished thinking. That breaks retries, timeouts, load balancing, moderation and billing —
   all at once.
3. **The bottleneck is memory bandwidth and HBM capacity**, not CPU or network. The mental
   model you brought from web services will point you at the wrong things.

### Functional requirements

- Send a message in a conversation and receive a streamed reply.
- Multi-turn: the model must see the conversation history.
- Persist conversations; list them; resume an old one.
- Regenerate / edit-and-resend a turn.
- Two tiers: **Free** (rate-limited, smaller model) and **Plus** (paid, priority, larger model).
- Content moderation on input and output.

### Non-functional requirements

- **Time-to-first-token (TTFT) < ~1 s at p95.** This is the latency users actually feel.
  Total response time is *not* the SLO — a 40-second answer that starts in 300 ms feels fast.
- **Inter-token latency steady at ≥ ~20–30 tok/s** — faster than a person reads, and no long
  stalls mid-sentence.
- **Graceful degradation under overload.** Rejecting a user in 50 ms is better than making
  them wait 90 s for a reply they've already given up on. Notebook 2 measures this.
- **Cost-aware.** Compute is the dominant COGS line, not storage or bandwidth. A design that
  is 30% cheaper per token is a strategically different company.

### Explicitly out of scope

Training, fine-tuning, RLHF, and the data pipeline behind them. This lab is about **inference
serving** — the thing that runs a million times a minute after training is done.


## 📏 Back-of-envelope capacity

Every number below is a **named variable** with a stated assumption. None of them are facts
about any real deployment — they are plausible values chosen so you can see the *method*, and
so you can change one and watch the conclusion move.

The order matters. We go: **demand → tokens → GPUs → memory → storage → money**, and we
sanity-check at each step with a number a human can eyeball.

### Step 1: demand model

The trap here is thinking in *requests*. Requests are not the unit of work — **tokens** are,
and the two are related by a factor that changes by 100× depending on how long conversations
get. So we model conversations, not requests.


In [ ]:
# ==== ASSUMPTIONS: demand ====
# All values are ASSUMPTIONS, not measurements. Plausible ranges are noted so you can see
# how sensitive the answer is. Change one and re-run the whole notebook.

DAU                      = 100_000_000  # daily active users        [range: 10M - 500M]
PLUS_FRACTION            = 0.05         # paid tier share of DAU    [range: 0.01 - 0.10]
CONV_PER_DAY_FREE        = 1.5          # conversations/day, free   [range: 0.5 - 3]
CONV_PER_DAY_PLUS        = 6.0          # conversations/day, plus   [range: 3 - 20]
TURNS_PER_CONVERSATION   = 6            # one turn = 1 user msg + 1 reply   [range: 2 - 20]

SYSTEM_PROMPT_TOKENS     = 400          # instructions prepended to every turn [range: 100 - 5000]
USER_MSG_TOKENS          = 60           # average user message      [range: 20 - 500]
ASSISTANT_REPLY_TOKENS   = 350          # average model reply       [range: 100 - 1500]

PEAK_TO_MEAN             = 2.0          # diurnal peak / daily mean [range: 1.5 - 4]

plus_users = DAU * PLUS_FRACTION
free_users = DAU - plus_users
conversations_per_day = free_users * CONV_PER_DAY_FREE + plus_users * CONV_PER_DAY_PLUS

print(f'Free users            : {free_users:>18,.0f}')
print(f'Plus users            : {plus_users:>18,.0f}')
print(f'Conversations / day   : {conversations_per_day:>18,.0f}')


### Step 2: conversations → tokens (the step everyone gets wrong)

Here is the thing that makes LLM capacity planning different from every other system you have
sized. **A chat model is stateless.** It has no memory between calls. To answer turn 6, the
server re-sends turns 1–5 *in full* as part of the prompt.

So the prompt does not stay the same size. It grows linearly with the turn number, and the
total prompt work over a conversation grows **quadratically**:

```
turn 1 prompt: [system][u1]
turn 2 prompt: [system][u1][a1][u2]
turn 3 prompt: [system][u1][a1][u2][a2][u3]
...
```

For a conversation of `T` turns with system prompt `S`, user message `u`, reply `r`:

```
prefill tokens = T·(S + u)  +  (u + r)·T·(T−1)/2     <-- quadratic in T
decode  tokens = T·r                                  <-- linear in T
```

If you model "tokens per turn" as a constant, you will undercount prompt processing by several
times over, and you will size the fleet for the wrong phase of the workload.


In [ ]:
# ==== Tokens per conversation: prefill (read) vs decode (write) ====
S = SYSTEM_PROMPT_TOKENS
u = USER_MSG_TOKENS
r = ASSISTANT_REPLY_TOKENS
T = TURNS_PER_CONVERSATION

# Sum over turns of: system + all prior messages + this user message
prefill_tokens_per_conv = T * (S + u) + (u + r) * T * (T - 1) // 2
decode_tokens_per_conv  = T * r

print('Per-turn prompt growth inside one conversation:')
for t in range(1, T + 1):
    prompt_t = S + (t - 1) * (u + r) + u
    print(f'  turn {t}: prompt = {prompt_t:>6,} tokens   reply = {r:>5,} tokens')

print()
print(f'Prefill tokens / conversation : {prefill_tokens_per_conv:>10,}')
print(f'Decode  tokens / conversation : {decode_tokens_per_conv:>10,}')
print(f'Prefill : decode ratio        : {prefill_tokens_per_conv/decode_tokens_per_conv:>10.1f} : 1')
print()
print('A naive "tokens per turn = u + r" model would have said'
      f' {T*(u+r):,} prompt-ish tokens;')
print(f'the real prompt work is {prefill_tokens_per_conv:,} — '
      f'{prefill_tokens_per_conv/(T*(u+r)):.1f}x more.')


In [ ]:
# ==== Aggregate token rates ====
prefill_tokens_per_day = conversations_per_day * prefill_tokens_per_conv
decode_tokens_per_day  = conversations_per_day * decode_tokens_per_conv
total_tokens_per_day   = prefill_tokens_per_day + decode_tokens_per_day

peak_prefill_tps = prefill_tokens_per_day / 86_400 * PEAK_TO_MEAN
peak_decode_tps  = decode_tokens_per_day  / 86_400 * PEAK_TO_MEAN

print(f'Prefill tokens / day  : {prefill_tokens_per_day:>18,.0f}')
print(f'Decode  tokens / day  : {decode_tokens_per_day:>18,.0f}')
print(f'Total   tokens / day  : {total_tokens_per_day:>18,.0f}')
print()
print(f'PEAK prefill tokens/s : {peak_prefill_tps:>18,.0f}')
print(f'PEAK decode  tokens/s : {peak_decode_tps:>18,.0f}')
print()
# ---- SANITY CHECK: a number a human can actually judge ----
words_per_token = 0.75
decode_words_per_user_per_day = decode_tokens_per_day / DAU * words_per_token
print('--- SANITY CHECK (eyeball this before trusting anything below) ---')
print(f'Generated text per user per day: {decode_tokens_per_day/DAU:,.0f} tokens '
      f'≈ {decode_words_per_user_per_day:,.0f} words ≈ '
      f'{decode_words_per_user_per_day/250:.1f} pages of A4.')
print('If that does not match your mental image of an average user, the demand model is')
print('wrong and every number after this point is wrong with it. Fix it here, not later.')


**Read that sanity check carefully.** ~2,700 generated words per user per day is roughly
11 pages. That is a *heavy* average. It is defensible only because the 5% Plus cohort does
4× the conversations of everyone else and drags the mean up — but if you believe the median
user asks one short question and leaves, `CONV_PER_DAY_FREE` and `ASSISTANT_REPLY_TOKENS` are
too high and the fleet below is oversized by 2–3×.

This is what back-of-envelope is *for*. Not precision — catching the assumption that is wrong
by an order of magnitude, before it becomes a hardware order.

### Step 3: tokens → GPUs

Now we need a throughput figure per accelerator. This is the single most load-bearing
assumption in the whole model, so it gets stated loudly:

> **ASSUMPTION.** A serving GPU sustains on the order of **2,000 decode tokens/second** for a
> mid-size dense model when it is well batched, and roughly **10× that for prefill**.
> Plausible range: 500–5,000 decode tok/s/GPU depending on model size, quantisation,
> accelerator generation and batch size.
>
> Nothing below depends on the number `2000` being right. It depends on the *shape*: prefill
> is far more efficient per token than decode, so the fleet is sized by decode. We sweep the
> assumption at the end of this section so you can see exactly how much it matters.

Why is prefill ~10× cheaper per token? Because prefill processes the whole prompt **in
parallel** as one big matrix multiply — it saturates the GPU's compute units. Decode produces
**one token at a time**, and every single token requires streaming the entire model's weights
out of HBM. Decode is memory-bandwidth-bound; prefill is compute-bound. Notebook 2 simulates
this directly.


In [ ]:
# ==== ASSUMPTIONS: accelerator throughput ====
DECODE_TOKENS_PER_SEC_PER_GPU  = 2_000     # ASSUMPTION [range: 500 - 5,000]
PREFILL_SPEEDUP                = 10        # ASSUMPTION: prefill tok/s / decode tok/s [range: 5 - 30]
PREFILL_TOKENS_PER_SEC_PER_GPU = DECODE_TOKENS_PER_SEC_PER_GPU * PREFILL_SPEEDUP

gpus_for_decode  = peak_decode_tps  / DECODE_TOKENS_PER_SEC_PER_GPU
gpus_for_prefill = peak_prefill_tps / PREFILL_TOKENS_PER_SEC_PER_GPU
gpus_compute     = gpus_for_decode + gpus_for_prefill

print(f'GPUs needed for decode  : {gpus_for_decode:>10,.0f}')
print(f'GPUs needed for prefill : {gpus_for_prefill:>10,.0f}')
print(f'GPUs (compute-bound)    : {gpus_compute:>10,.0f}')
print()
print(f'Note the inversion: prefill is {prefill_tokens_per_day/decode_tokens_per_day:.1f}x more TOKENS '
      f'but only {gpus_for_prefill/gpus_compute:.0%} of the')
print(f'GPUs, because each prefill token is ~{PREFILL_SPEEDUP}x cheaper to process.')
print('Decode dominates the fleet. Optimise decode first.')
print()
print('--- SANITY CHECK ---')
print(f'One GPU serves {DAU/gpus_compute:,.0f} daily active users.')
print('Is that plausible? If a GPU appeared to serve 10 million users, the throughput')
print('assumption is off by 1000x. If it served 50, you have priced a supercomputer per user.')


### Step 4: the KV cache — the constraint that is actually binding

Here is the part that has no analogue in a normal web service.

While the model generates a reply, it keeps a **KV cache**: for every token in the context, for
every layer, it stores the key and value vectors so it doesn't have to recompute attention over
the whole history at each step. That cache lives in GPU HBM, it is **per active conversation**,
and it grows with every token generated.

```
KV bytes per token = 2 (K and V) × layers × kv_heads × head_dim × bytes_per_element
```

This is why "how many users can one GPU serve concurrently" is a *memory* question, not a
compute question. Let's find out which one binds.

First: how many generations are in flight at peak? That's Little's Law —
`concurrency = arrival rate × time in system`.


In [ ]:
# ==== ASSUMPTIONS: model shape and accelerator memory ====
N_LAYERS            = 64      # ASSUMPTION [range: 32 - 126]
N_KV_HEADS          = 8       # ASSUMPTION: grouped-query attention [range: 8 - 64]
HEAD_DIM            = 128     # ASSUMPTION [range: 64 - 256]
KV_BYTES_PER_ELEMENT = 2      # fp16/bf16 KV cache [1 if fp8-quantised]

MODEL_PARAMS_B      = 70      # billions of parameters   [range: 8 - 700]
WEIGHT_BYTES_PARAM  = 2       # bf16 weights             [1 if int8, 0.5 if int4]
GPU_HBM_GB          = 80      # per accelerator          [range: 40 - 192]
TENSOR_PARALLEL     = 4       # GPUs the model is sharded over
ACTIVATION_GB       = 5       # workspace/activations reserved per GPU [range: 2 - 15]

TARGET_TOKENS_PER_SEC_PER_USER = 30   # perceived streaming speed [range: 15 - 100]

# --- Little's Law: how many generations are alive at once? ---
turns_per_sec_peak   = conversations_per_day * T / 86_400 * PEAK_TO_MEAN
generation_seconds   = r / TARGET_TOKENS_PER_SEC_PER_USER
concurrent_generations = turns_per_sec_peak * generation_seconds

print(f'Turns / sec at peak            : {turns_per_sec_peak:>14,.0f}')
print(f'Seconds to stream one reply    : {generation_seconds:>14,.1f}')
print(f'Concurrent generations (peak)  : {concurrent_generations:>14,.0f}')
print()
# Cross-check: this must reproduce the decode rate we derived independently.
crosscheck = concurrent_generations * TARGET_TOKENS_PER_SEC_PER_USER
print(f'Cross-check  concurrency x tok/s/user = {crosscheck:>14,.0f} decode tok/s')
print(f'             derived earlier          = {peak_decode_tps:>14,.0f} decode tok/s')
print(f'             agree? {abs(crosscheck - peak_decode_tps) < 1:}   '
      '(two independent routes to the same number)')


In [ ]:
# ==== KV cache memory ====
kv_bytes_per_token = 2 * N_LAYERS * N_KV_HEADS * HEAD_DIM * KV_BYTES_PER_ELEMENT

# An active generation holds: its whole prompt, plus the tokens generated so far.
# Averaged over the life of the request, "so far" is about half the reply.
mean_prompt_tokens = prefill_tokens_per_conv / T
avg_resident_tokens = mean_prompt_tokens + r / 2

kv_bytes_per_request = avg_resident_tokens * kv_bytes_per_token
kv_bytes_total       = concurrent_generations * kv_bytes_per_request

# How much HBM is actually free for KV, after weights and workspace?
weights_gb_total   = MODEL_PARAMS_B * WEIGHT_BYTES_PARAM
weights_gb_per_gpu = weights_gb_total / TENSOR_PARALLEL
kv_budget_gb_per_gpu = GPU_HBM_GB - weights_gb_per_gpu - ACTIVATION_GB

print(f'KV bytes / token              : {kv_bytes_per_token:>12,}  '
      f'({kv_bytes_per_token/1024:.0f} KiB per token!)')
print(f'Mean prompt at generation time: {mean_prompt_tokens:>12,.0f} tokens')
print(f'Avg resident context          : {avg_resident_tokens:>12,.0f} tokens')
print(f'KV per active conversation    : {kv_bytes_per_request/1e6:>12,.0f} MB')
print()
print(f'Model weights (total)         : {weights_gb_total:>12,.0f} GB '
      f'-> {weights_gb_per_gpu:,.0f} GB/GPU at TP={TENSOR_PARALLEL}')
print(f'HBM left for KV per GPU       : {kv_budget_gb_per_gpu:>12,.0f} GB')
print(f'Aggregate KV needed at peak   : {kv_bytes_total/1e12:>12,.0f} TB')
print()
gpus_for_kv = kv_bytes_total / (kv_budget_gb_per_gpu * 1e9)
print(f'GPUs needed just to HOLD the KV cache : {gpus_for_kv:>10,.0f}')
print(f'GPUs needed for compute               : {gpus_compute:>10,.0f}')
fleet_peak_gpus = max(gpus_for_kv, gpus_compute)
binding = 'COMPUTE' if gpus_compute >= gpus_for_kv else 'KV MEMORY'
print(f'-> fleet at peak = max(...)           : {fleet_peak_gpus:>10,.0f}   binding: {binding}')


**~435 MB of HBM per open conversation**, with the assumptions above. That is the number to
remember. An 80 GB accelerator holding 35 GB of weights has ~40 GB left, which is roughly
**90 concurrent conversations per GPU** — and that is at a modest ~1,660-token average
context. Let one user paste a 100k-token document and that single conversation needs
`100_000 x 256 KiB ≈ 26 GB`, a *third of the card*, for one person.

Now, which constraint binds? With these assumptions, compute — but only by ~2×, and that
margin is entirely at the mercy of the throughput assumption we flagged. Let's sweep it.


In [ ]:
# ==== Sensitivity: which constraint binds, as throughput assumption varies? ====
print(f'{"decode tok/s/GPU":>17} | {"GPUs (compute)":>15} | {"GPUs (KV)":>10} | '
      f'{"fleet":>8} | binds')
print('-' * 72)
for tps in (500, 1_000, 2_000, 3_000, 4_000, 5_000, 8_000):
    g_dec = peak_decode_tps / tps
    g_pre = peak_prefill_tps / (tps * PREFILL_SPEEDUP)
    g_com = g_dec + g_pre
    fleet = max(g_com, gpus_for_kv)
    print(f'{tps:>17,} | {g_com:>15,.0f} | {gpus_for_kv:>10,.0f} | {fleet:>8,.0f} | '
          f'{"compute" if g_com >= gpus_for_kv else "KV MEMORY"}')

print()
print('The crossover sits between 3,000 and 5,000 decode tok/s/GPU. Above it, making the')
print('GPU faster buys you NOTHING — you run out of memory to hold conversations before you')
print('run out of arithmetic. That is the moment KV-cache engineering (paged attention,')
print('fp8 KV, prefix sharing, offload to CPU) stops being an optimisation and becomes the')
print('only lever you have left.')


⚖️ **What this model is still hiding:**

- **`avg_resident_tokens` is a mean over a savage distribution.** Context lengths are
  heavy-tailed: most turns are ~1.5k tokens, a few are 200k. The mean sizes your fleet; the
  tail decides whether a single request can OOM a GPU. Real schedulers need a per-request
  context cap for exactly this reason.
- **We assumed every conversation runs on the same model.** In reality Free gets a smaller,
  cheaper model and Plus gets the large one — two fleets, two cost structures, and a routing
  decision. Splitting the model above by tier changes the GPU count more than any other
  single edit.
- **Prefill and decode are averaged into one fleet.** They have opposite bottlenecks, so
  production systems increasingly run them on *separate* pools ("disaggregated serving") and
  ship the KV cache between them over the network. That trades interconnect bandwidth for
  better utilisation of both phases.
- **No prefix caching.** We charge full prefill for every turn. Notebook 3 shows that reusing
  the KV of the shared conversation prefix removes ~70% of prefill work — which would cut
  `gpus_for_prefill` by the same factor.

### Step 5: conversation history storage

After the GPU numbers, storage is almost a rounding error — but it is the part that grows
forever, so it's worth a line.


In [ ]:
# ==== ASSUMPTIONS: storage ====
CHARS_PER_TOKEN      = 4      # UTF-8 bytes per token, English text [range: 3 - 5]
BYTES_META_PER_MSG   = 200    # ids, role, timestamps, model, finish_reason, token counts
REPLICATION_FACTOR   = 3      # durable store replicas
RETENTION_YEARS      = 3

messages_per_conv    = 2 * T
text_tokens_per_conv = T * u + T * r          # what we PERSIST is the text, once — not the
                                              # re-sent prompt, which is reconstructed at read
bytes_per_conv = text_tokens_per_conv * CHARS_PER_TOKEN + messages_per_conv * BYTES_META_PER_MSG

tb_per_day   = conversations_per_day * bytes_per_conv / 1e12
tb_per_year  = tb_per_day * 365
pb_retained  = tb_per_year * RETENTION_YEARS * REPLICATION_FACTOR / 1_000

print(f'Stored text per conversation  : {text_tokens_per_conv:>10,} tokens '
      f'-> {bytes_per_conv/1024:,.1f} KiB')
print(f'Conversation writes / sec (pk): {conversations_per_day/86_400*PEAK_TO_MEAN:>10,.0f}')
print(f'Message writes / sec (peak)   : {conversations_per_day*messages_per_conv/86_400*PEAK_TO_MEAN:>10,.0f}')
print(f'Raw storage / day             : {tb_per_day:>10,.2f} TB')
print(f'Raw storage / year            : {tb_per_year:>10,.0f} TB')
print(f'{REPLICATION_FACTOR}x replicated, {RETENTION_YEARS}y kept    : {pb_retained:>10,.1f} PB')
print()
print('--- SANITY CHECK ---')
print(f'Per user per day: {bytes_per_conv*conversations_per_day/DAU/1024:,.0f} KiB. '
      'A few pages of text. Sounds right.')
print()
print('Note what storage is NOT: it is not the prompt. We store each message once and')
print('rebuild the prompt at read time. Storing the fully-expanded prompt for every turn')
print(f'would cost {prefill_tokens_per_conv/text_tokens_per_conv:.1f}x more for zero benefit.')


The storage design falls straight out of the access pattern:

| Access pattern | Frequency | Store |
|---|---|---|
| Append a message to conversation `c` | every turn | wide-column / LSM store, partition key `conversation_id`, clustering key `created_at` |
| Read the last N messages of `c` | every turn | same store — this is a single-partition range scan, the cheapest thing it does |
| List a user's conversations | on app open | separate index table keyed by `user_id`, holding only `(conversation_id, title, updated_at)` |
| Search across all conversations | rare | offline index (OpenSearch), fed asynchronously |

There is no join, no transaction spanning conversations, and no read-modify-write. That is a
Cassandra/DynamoDB-shaped workload, not a Postgres-shaped one — and the reason is simply that
the entity boundary (`conversation_id`) is also the only access boundary.

### Step 6: cost — the number that decides the product


In [ ]:
# ==== ASSUMPTIONS: cost ====
GPU_COST_PER_HOUR   = 3.00   # fully-loaded $/GPU-hour (amortised HW + power + DC)
                             # ASSUMPTION [range: 1.50 - 12.00 depending on buy vs rent]
TARGET_UTILISATION  = 0.70   # you cannot run a fleet at 100% [range: 0.5 - 0.85]
PLUS_PRICE_PER_MONTH = 20.0  # ASSUMPTION: paid-tier subscription price

provisioned_gpus = fleet_peak_gpus / TARGET_UTILISATION
annual_gpu_cost  = provisioned_gpus * GPU_COST_PER_HOUR * 24 * 365
tokens_per_year  = total_tokens_per_day * 365
output_tokens_per_year = decode_tokens_per_day * 365

cost_per_m_all_tokens    = annual_gpu_cost / (tokens_per_year / 1e6)
cost_per_m_output_tokens = annual_gpu_cost / (output_tokens_per_year / 1e6)

print(f'Peak fleet                    : {fleet_peak_gpus:>14,.0f} GPUs')
print(f'Provisioned at {TARGET_UTILISATION:.0%} utilisation : {provisioned_gpus:>14,.0f} GPUs')
print(f'Annual compute cost           : ${annual_gpu_cost/1e6:>13,.0f} M')
print()
print(f'Cost per 1M tokens (all)      : ${cost_per_m_all_tokens:>13,.3f}')
print(f'Cost per 1M OUTPUT tokens     : ${cost_per_m_output_tokens:>13,.2f}   '
      '<-- compare to public API prices')
print()
print('--- SANITY CHECK: does the business close? ---')
plus_revenue = plus_users * PLUS_PRICE_PER_MONTH * 12
print(f'Plus subscription revenue     : ${plus_revenue/1e9:>13,.2f} B/yr')
print(f'Serving compute               : ${annual_gpu_cost/1e9:>13,.2f} B/yr '
      f'({annual_gpu_cost/plus_revenue:.0%} of subscription revenue)')
print()
print('That ratio is the whole product strategy in one number. Serving compute eating')
print(f'{annual_gpu_cost/plus_revenue:.0%} of subscription revenue leaves room for salaries, training runs and')
print('margin. At 90% it does not, and you must either raise prices, shrink the model for')
print('the free tier, or cut tokens per user. Notebook 3 shows the levers.')


In [ ]:
# ==== Where does the money actually go? ====
# Split the bill by phase, using the GPU split we derived.
share_decode  = gpus_for_decode / gpus_compute
share_prefill = gpus_for_prefill / gpus_compute

print('Cost attribution by phase (at the compute-bound fleet split):')
print(f'  decode  : {share_decode:>5.0%}  of spend, '
      f'{decode_tokens_per_day/total_tokens_per_day:>5.0%} of tokens')
print(f'  prefill : {share_prefill:>5.0%}  of spend, '
      f'{prefill_tokens_per_day/total_tokens_per_day:>5.0%} of tokens')
print()
print('This is exactly why every LLM API prices INPUT and OUTPUT tokens differently, at')
print('roughly the ratio you see above. If you charged one flat price per token, users')
print('with long prompts and short answers would be subsidised by everyone else.')
print()
ratio = (share_decode / (decode_tokens_per_day/total_tokens_per_day)) / \
        (share_prefill / (prefill_tokens_per_day/total_tokens_per_day))
print(f'  implied output:input price ratio = {ratio:.1f} : 1')
print()
print('Be honest about what just happened: that ratio is exactly PREFILL_SPEEDUP, because')
print('we defined cost per token as inversely proportional to throughput. It is a tautology,')
print('not a discovery. The non-obvious part is the DIRECTION of the argument: the price')
print('ratio a vendor charges you is a public estimate of their internal prefill:decode')
print('throughput ratio. Look up two real API prices and you have measured their hardware.')


## 🧱 High-level architecture

```
   ┌──────────┐   HTTPS POST /v1/chat  (+ SSE / WebSocket response)
   │  Client  │◀──────────────────────────────────────────────┐
   └────┬─────┘                                               │
        │                                                     │ token stream
        ▼                                                     │
   ┌───────────────┐   authn, tier, quota                     │
   │  API gateway  │──────────────┐                           │
   └────┬──────────┘              ▼                           │
        │                  ┌──────────────┐                   │
        │                  │ Token meter  │ (Redis: weighted  │
        │                  │  + quotas    │  token buckets)   │
        │                  └──────────────┘                   │
        ▼                                                     │
   ┌────────────────┐   moderate input (fast classifier)      │
   │  Orchestrator  │──▶ ┌──────────────┐                     │
   │  (stateful per │    │  Moderation  │                     │
   │   generation)  │◀── └──────────────┘                     │
   └───┬────────┬───┘                                         │
       │        │  build prompt                               │
       │        ▼                                             │
       │   ┌───────────────────┐   ┌──────────────────┐       │
       │   │ Conversation store│   │  Prompt / result │       │
       │   │  (wide-column)    │   │  caches (Redis)  │       │
       │   └───────────────────┘   └──────────────────┘       │
       │                                                      │
       ▼  enqueue (priority: plus > free)                      │
   ┌──────────────────────────┐                               │
   │   Inference request queue │  bounded! sheds load          │
   └───────────┬──────────────┘                               │
               ▼                                              │
   ┌──────────────────────────────────────────┐               │
   │  Scheduler / router                      │               │
   │   - continuous batching                  │               │
   │   - KV-cache admission control           │               │
   │   - prefix-cache-aware placement         │               │
   └───┬──────────────┬──────────────┬────────┘               │
       ▼              ▼              ▼                        │
   ┌────────┐    ┌────────┐    ┌────────┐                     │
   │Replica │    │Replica │    │Replica │  … N GPU replicas   │
   │(TP=4)  │    │(TP=4)  │    │(TP=4)  │                     │
   │ KV$    │    │ KV$    │    │ KV$    │                     │
   └───┬────┘    └────────┘    └────────┘                     │
       │ tokens                                               │
       ▼                                                      │
   ┌───────────────────────┐  buffered window, seq-numbered   │
   │ Streaming relay       │──────────────────────────────────┘
   │  + output moderation  │
   │  + resumable token log│
   └───────────────────────┘
```

The pieces that would *not* appear in a normal chat-app diagram, and why each is here:

- **Token meter before the queue.** Quota has to be checked before you spend a GPU-second,
  and it must be denominated in tokens, not requests. (Notebook 3 shows why.)
- **A bounded queue.** An unbounded queue in front of a fixed-capacity GPU fleet is a
  latency bomb. (Notebook 2 measures it.)
- **A scheduler that knows about KV memory.** Admission is a memory-allocation decision, not
  a "is there a free worker" decision.
- **A streaming relay with a token log.** The response is delivered incrementally, so
  retries, resumes, moderation and billing all need a durable record of *what was already
  sent*. (Notebook 3.)


## ✅ Summary

- **Tokens, not requests, are the unit of work.** Requests vary by 100× in cost; sizing on
  request rate is how you end up 10× wrong.
- **Prompt work grows quadratically with conversation length**, because a stateless model
  re-reads the whole history every turn. This is the single biggest term in the token budget.
- **Prefill and decode are different systems** sharing a chip: prefill is compute-bound and
  cheap per token, decode is memory-bandwidth-bound and expensive. Decode sizes the fleet.
- **The KV cache is a real, hard memory constraint** — hundreds of MB per open conversation.
  Whether compute or memory binds depends on assumptions that are only good to ±2×, so a
  serious design measures both and re-checks after every model change.
- **Cost per million tokens follows from the fleet, not from a price list**, and the
  input/output cost split explains why every LLM API prices them separately.

Next: [Notebook 2 — Serving & Scaling](./02_serving_and_scaling.ipynb), where we build the
scheduler and watch naive request handling fall over.
